# Introduction to Generative Language Modelling

How do machines generate text? This notebook introduces the core idea behind modern LLMs: **next-word prediction**.

## The Core Idea: Predict the Next Word

All modern language models — from simple n-grams to GPT-4 — share the same fundamental approach:

> Given some context, predict the most likely next word.

```
"The cat sat on the ___"  →  "mat" (most likely)
```

The difference between simple and powerful models is:
- **How much context** they can consider
- **How well** they learn patterns from data

## Hands-On: Building a Simple Language Model

Let's build a basic "language model" using word statistics. This illustrates the core concept before we move to neural networks.

In [ ]:
import random
from collections import Counter
from IPython.display import display, Markdown

# A simple training corpus
corpus = """
The cat sat on the mat. The dog sat on the rug. 
The cat chased the mouse. The dog chased the cat.
The mouse ran away. The cat ran after the mouse.
"""

display(Markdown(corpus))


In [ ]:
# Tokenize into words
words = corpus.lower().split()

words

### Step 1: Learn Word Patterns (Bigrams)

A **bigram** model learns: "Given word A, what word typically comes next?"

In [ ]:
# Build bigram model: for each word, count what follows it
bigrams = {}
for i in range(len(words) - 1):
    current_word = words[i]
    next_word = words[i + 1]
    
    if current_word not in bigrams:
        bigrams[current_word] = []
    bigrams[current_word].append(next_word)

# What words typically follow certain words?
{
    "the →": bigrams.get("the", []),
    "cat →": bigrams.get("cat", []),
    "dog →": bigrams.get("dog", []),
}

### Step 2: Generate Text by Sampling

To generate text, we:
1. Start with a word
2. Randomly pick a word that typically follows it
3. Repeat

In [ ]:
def generate_text(start_word, num_words=10):
    """Generate text using our bigram model."""
    result = [start_word]
    current = start_word
    
    for _ in range(num_words - 1):
        if current in bigrams and bigrams[current]:
            # Randomly sample from words that follow current word
            next_word = random.choice(bigrams[current])
            result.append(next_word)
            current = next_word
        else:
            break
    
    return " ".join(result)

# Generate a few examples — run this cell multiple times!
[generate_text("the") for _ in range(3)]

**Observation:** The generated text is more or less grammatical but not very coherent. Why?

- The model only looks at **one word** of context
- It has no understanding of meaning, just statistics
- The training data is tiny

**Key insight:** Better language models use more context and learn deeper patterns.

## From Statistics to Neural Networks

| Approach | Context | Learning |
|----------|---------|----------|
| **Bigrams** | 1 word | Simple counting |
| **N-grams** | N-1 words | Simple counting |
| **RNNs** (2010s) | ~100 words | Neural network with memory |
| **Transformers** (2017+) | 1000s of words | Large neural network with _attention mechanism_ |

The **Transformer** architecture (GPT, Claude, Llama) revolutionized NLP by:
1. Processing the **entire context at once** (not word-by-word)
2. Using **attention** to focus on relevant parts of the context
3. Scaling to **billions of parameters** trained on massive datasets

## The Transformer Intuition

Unlike RNNs that process words one-by-one, transformers see the **whole context at once**:

```
Input:  "She was talking to"
         ↓    ↓     ↓     ↓
      [======================]
      [   Transformer Model  ]  ← Sees all words simultaneously
      [======================]
                ↓
Output: "him" (predicted next word)
```

The **attention mechanism** learns which words in the context are most important for predicting the next word.

## Emergent Capabilities: Scale is (almost) All You Need

A surprising discovery: when you train transformers on **massive data** with **billions of parameters**, capabilities emerge that weren't explicitly trained:

| Model Size | Emergent Capabilities |
|------------|----------------------|
| Small | Basic text completion |
| Medium | Grammar, style matching |
| Large | Reasoning, code generation, translation |
| Very Large | Complex reasoning, multi-step problem solving |

This is why GPT-4 can write code, translate languages, and answer complex questions — even though it was "just" trained to predict the next word!

## Zero-Shot, Few-Shot: The Power of Prompting

Traditional ML required training a separate model for each task. Modern LLMs can handle new tasks through **prompting**:

| Approach | Example |
|----------|---------|
| **Zero-shot** | "Translate to French: Hello" → "Bonjour" |
| **One-shot** | "sea → mer, Hello → ?" → "Bonjour" |
| **Few-shot** | Give 3-5 examples, then ask for new one |

No retraining needed — the model generalizes from its pre-training!

## Hands-On: See It In Action with Ollama

Let's see a real language model predict the next word:

In [ ]:
import ollama
from IPython.display import Markdown

# Use a small model for quick demos
model = "tinyllama:latest"

def complete_text(prompt, max_tokens):
    """Complete text using an LLM."""
    response = ollama.generate(
        model=model,
        prompt=prompt,
        raw=True,
        options={"num_predict": max_tokens, "temperature": 0.7}
    )
    return response["response"]

def show_completion(prompt, max_tokens=10):
    """Display a prompt and its completion nicely."""
    completion = complete_text(prompt, max_tokens)
    return Markdown(f"**Prompt:** {prompt}...\n\n**→** {completion}")

show_completion("The capital of France is", 10)

## Exercise: Experiment with Prompts

Try different prompts and observe how the model completes them:

In [ ]:
# Try your own prompts!
show_completion("Once upon a time, there was a", 100)

## Key Takeaways

1. **Next-word prediction** is the foundation of all language models
2. **More context + better learning** = more capable models
3. **Transformers** process entire contexts at once using attention
4. **Scale** leads to emergent capabilities (reasoning, code, etc.)
5. **Prompting** lets us use one model for many tasks

---
_This notebook is licensed under a [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0)](https://creativecommons.org/licenses/by-nc-sa/4.0/). Copyright © 2025-2026 [Christian Staudt](https://clstaudt.me) and [Katharina Rasch](https://krasch.io)_